In [ ]:
%pip install -q "transformers>=4.55,<5" accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json, torch
m = AutoModelForCausalLM.from_pretrained("ujwal00/qwen3-1.7b-json", dtype=torch.float16, device_map="auto")
tok = AutoTokenizer.from_pretrained("ujwal00/qwen3-1.7b-json")
SYSTEM = "You are a precise JSON generator. Given a JSON Schema, output ONE JSON value that strictly validates against it. Output ONLY the JSON value."
tests = [
  {"type":"object","properties":{"name":{"type":"string"},"age":{"type":"integer"}},"required":["name","age"]},
  {"type":"object","properties":{"user":{"type":"object","properties":{"id":{"type":"integer"},"email":{"type":"string","format":"email"},"roles":{"type":"array","items":{"type":"string","enum":["admin","user"]}}},"required":["id","email"]}},"required":["user"]},
]
for schema in tests:
    user = "Generate a JSON value that validates against this JSON Schema:\n\n" + json.dumps(schema) + "\n\nReturn only the JSON value."
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(m.device)
    out = m.generate(ids, max_new_tokens=512, do_sample=False)
    print("SCHEMA:", json.dumps(schema)[:130])
    print("OUTPUT:", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print("=" * 70)
